# 02 · Aquifer recharge matching — store the flood underground

**Idea:** instead of (only) surface tanks, find places where flood-surplus water sits directly
above a *depleted, permeable* aquifer. Recharging groundwater there stores monsoon water for
the dry season with zero evaporation and zero land acquisition.

**Recharge Suitability Index (RSI)** per pixel =
weighted mix of: depth to water table (deeper = more empty storage), soil permeability
(sandier = faster intake), pervious surface fraction, and water availability (flow accumulation).
We then score each sink from notebook 01 as `volume x RSI` and rank recharge sites.

**Requires:** notebook 01 outputs in the same `WORK` folder.

In [ ]:
import sys, subprocess, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("rasterio")

WORK = "/content/drive/MyDrive/floodtwin" if os.path.isdir("/content/drive/MyDrive") else "/content/floodtwin"
assert os.path.exists(f"{WORK}/sinks.csv"), "Run notebook 01 first."

import ee
PROJECT_ID = "your-cloud-project-id"   # <-- EDIT
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

AOI = (85.02, 25.54, 85.30, 25.68)
region = ee.Geometry.Rectangle(list(AOI))
SCALE = 30

## Groundwater levels — the one dataset you must fetch manually
CGWB observation-well data lives on **India-WRIS** (https://indiawris.gov.in → Groundwater
Level). Download pre-monsoon depth-to-water (metres below ground) for **Patna district**
stations as CSV with columns: `station,lat,lon,depth_to_water_m`.

The cell below writes a **SAMPLE file with made-up values** so the pipeline runs end-to-end.
⚠️ **Replace `gw_levels.csv` with real CGWB data before believing any output.** Patna sits on
the prolific Gangetic alluvial aquifer, so real values matter enormously.

In [ ]:
import pandas as pd

gw_path = f"{WORK}/gw_levels.csv"
if not os.path.exists(gw_path):
    sample = pd.DataFrame([
        # SAMPLE / PLACEHOLDER VALUES — NOT REAL MEASUREMENTS. Replace with India-WRIS export.
        ("SAMPLE_Danapur",      25.633, 85.046, 8.5),
        ("SAMPLE_Phulwari",     25.575, 85.080, 7.2),
        ("SAMPLE_PatnaSadar",   25.610, 85.140, 6.0),
        ("SAMPLE_Sampatchak",   25.555, 85.180, 9.1),
        ("SAMPLE_PatnaCityE",   25.595, 85.230, 5.4),
        ("SAMPLE_Fatuha",       25.560, 85.290, 10.3),
    ], columns=["station", "lat", "lon", "depth_to_water_m"])
    sample.to_csv(gw_path, index=False)
    print("WROTE SAMPLE gw_levels.csv — replace with real CGWB/India-WRIS data!")

gw = pd.read_csv(gw_path)
print(gw)

In [ ]:
# Interpolate station depths to the raster grid (inverse-distance weighting).
import numpy as np
import rasterio

with rasterio.open(f"{WORK}/depth.tif") as src:
    depth = src.read(1); transform = src.transform; R, C = src.height, src.width

cols = np.arange(C) + 0.5; rows = np.arange(R) + 0.5
lons = transform.c + cols * transform.a
lats = transform.f + rows * transform.e
LON, LAT = np.meshgrid(lons, lats)

def idw(LON, LAT, pts, vals, power=2.0):
    num = np.zeros(LON.shape); den = np.zeros(LON.shape)
    for (plat, plon), v in zip(pts, vals):
        d2 = (LON - plon) ** 2 + (LAT - plat) ** 2 + 1e-12
        w = 1.0 / d2 ** (power / 2)
        num += w * v; den += w
    return num / den

gw_depth = idw(LON, LAT, gw[["lat", "lon"]].values, gw["depth_to_water_m"].values)
print("interpolated depth-to-water range (m):", gw_depth.min().round(1), "-", gw_depth.max().round(1))

## Soil permeability from SoilGrids
Sand/clay fractions (0–30 cm) → saturated hydraulic conductivity via the Cosby (1984)
pedotransfer function. Crude but consistent across the city; replace with NBSS&LUP maps later.

In [ ]:
import requests

def download_ee_image(img, path, scale=SCALE, bands=None):
    if bands is not None:
        img = img.select(bands)
    url = img.getDownloadURL({"region": region, "scale": scale,
                              "format": "GEO_TIFF", "crs": "EPSG:4326"})
    r_ = requests.get(url, timeout=600); r_.raise_for_status()
    open(path, "wb").write(r_.content); print("saved", path)

sand_img = ee.Image("projects/soilgrids-isric/sand_mean").select(
    ["sand_0-5cm_mean", "sand_5-15cm_mean", "sand_15-30cm_mean"]).reduce(ee.Reducer.mean())
clay_img = ee.Image("projects/soilgrids-isric/clay_mean").select(
    ["clay_0-5cm_mean", "clay_5-15cm_mean", "clay_15-30cm_mean"]).reduce(ee.Reducer.mean())
download_ee_image(sand_img, f"{WORK}/sand.tif")
download_ee_image(clay_img, f"{WORK}/clay.tif")

def read_band(p):
    with rasterio.open(p) as s:
        a = s.read(1).astype("float64")
    return a[:R, :C] if a.shape[0] >= R and a.shape[1] >= C else np.pad(
        a, ((0, R - a.shape[0]), (0, C - a.shape[1])), mode="edge")

sand_pct = read_band(f"{WORK}/sand.tif") / 10.0     # SoilGrids stores g/kg -> %
clay_pct = read_band(f"{WORK}/clay.tif") / 10.0
# Cosby 1984: log10 Ksat [inch/hr] = -0.6 + 0.0126*sand% - 0.0064*clay%
ksat_mm_hr = (10 ** (-0.6 + 0.0126 * sand_pct - 0.0064 * clay_pct)) * 25.4
print("Ksat range (mm/hr):", ksat_mm_hr.min().round(2), "-", ksat_mm_hr.max().round(2))

## Pervious fraction and the suitability index

In [ ]:
from scipy import ndimage

wc = read_band(f"{WORK}/worldcover.tif")
pervious = (~np.isin(wc, [50, 80])).astype("float64")          # not built-up, not water
pervious = ndimage.uniform_filter(pervious, size=7)            # neighbourhood fraction

acc = read_band(f"{WORK}/acc.tif")
avail = np.log1p(acc)                                          # water availability proxy

def norm(a, lo=2, hi=98):
    a = a.astype("float64")
    p1, p2 = np.nanpercentile(a, lo), np.nanpercentile(a, hi)
    return np.clip((a - p1) / max(p2 - p1, 1e-9), 0, 1)

W_GW, W_KS, W_PERV, W_AVAIL = 0.35, 0.30, 0.20, 0.15
rsi = (W_GW * norm(gw_depth) + W_KS * norm(ksat_mm_hr)
       + W_PERV * pervious + W_AVAIL * norm(avail))
rsi = np.where(wc == 80, 0, rsi)                               # exclude open water

meta = dict(driver="GTiff", height=R, width=C, count=1, crs="EPSG:4326",
            transform=transform, dtype="float32")
with rasterio.open(f"{WORK}/rsi.tif", "w", **meta) as dst:
    dst.write(rsi.astype("float32"), 1)

## Rank recharge sites = sink volume × local suitability

In [ ]:
sinks = pd.read_csv(f"{WORK}/sinks.csv")

def rsi_at(row, col, win=3):
    r0, r1 = max(0, row - win), min(R, row + win + 1)
    c0, c1 = max(0, col - win), min(C, col + win + 1)
    return float(np.nanmean(rsi[r0:r1, c0:c1]))

sinks["rsi"] = [rsi_at(int(r_), int(c_)) for r_, c_ in zip(sinks.row, sinks.col)]
sinks["gw_depth_m"] = [float(gw_depth[int(r_), int(c_)]) for r_, c_ in zip(sinks.row, sinks.col)]
sinks["ksat_mm_hr"] = [float(ksat_mm_hr[int(r_), int(c_)]) for r_, c_ in zip(sinks.row, sinks.col)]
sinks["recharge_score"] = sinks.volume_m3 * sinks.rsi
ranked = sinks.sort_values("recharge_score", ascending=False).reset_index(drop=True)
ranked.to_csv(f"{WORK}/recharge_sites.csv", index=False)
print(ranked.head(15)[["sink_id", "lat", "lon", "volume_m3", "gw_depth_m",
                       "ksat_mm_hr", "rsi", "recharge_score"]].to_string())

In [ ]:
import folium
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from PIL import Image

norm_v = mcolors.Normalize(vmin=0, vmax=1)
rgba = cm.YlGn(norm_v(rsi)); rgba[..., 3] = 0.55
Image.fromarray((rgba * 255).astype(np.uint8)).save(f"{WORK}/rsi_overlay.png")

lon0, lat1 = transform * (0, 0); lon1, lat0 = transform * (C, R)
m = folium.Map(location=[(lat0 + lat1) / 2, (lon0 + lon1) / 2], zoom_start=12, tiles="cartodbpositron")
folium.raster_layers.ImageOverlay(f"{WORK}/rsi_overlay.png",
                                  bounds=[[lat0, lon0], [lat1, lon1]]).add_to(m)
for _, s in ranked.head(15).iterrows():
    folium.CircleMarker([s.lat, s.lon], radius=7, color="darkgreen", fill=True,
        popup=(f"sink {int(s.sink_id)}<br>score {s.recharge_score:,.0f}"
               f"<br>water table {s.gw_depth_m:.1f} m down"
               f"<br>Ksat {s.ksat_mm_hr:.1f} mm/hr")).add_to(m)
m.save(f"{WORK}/recharge_map.html")
m

## Reading the result + caveats
- Top-ranked sites are where a recharge pit / injection well / percolation tank converts a
  flooding liability into dry-season groundwater. This is the framing used by Atal Bhujal
  Yojana and AMRUT — a ranked, costed list is exactly what those programmes fund.
- **The RSI is only as good as `gw_levels.csv`** — replace the sample file with real
  pre-monsoon CGWB data, and ideally add post-monsoon values to estimate aquifer response.
- Shallow aquifers in parts of Bihar carry arsenic; recharge siting near drinking-water wells
  needs water-quality screening. Final sites need a geotechnical + quality investigation —
  this notebook prioritises, it does not certify.